In [1]:
import pandas as pd

df = pd.read_csv("1 DataSet Residuos municipales generados anualmente.csv",encoding="ISO-8859-1",delimiter=";",dtype={"UBIGEO":str})
# Si el dataset se lee con encoding UTF-8 , generará error, por ello se utilizó otro encoding como ISO-8859-1
# El dataset tiene el campo UBIGEO que empieza con 0... , para evitar que pandas lo considere un int o float colocamos dtype

# Los datos han sido obtenidos de: https://www.datosabiertos.gob.pe/dataset/residuos-municipales-generados-anualmente

df.head()

,FECHA_CORTE,N_SEC,UBIGEO,REG_NAT,DEPARTAMENTO,PROVINCIA,DISTRITO,POB_TOTAL,POB_URBANA,POB_RURAL,GPC_DOM,QRESIDUOS_DOM,QRESIDUOS_NO_DOM,QRESIDUOS_MUN,PERIODO
0,20230614,1,10101,SELVA,AMAZONAS,CHACHAPOYAS,CHACHAPOYAS,28423,27548,875,0.48,4857.50,2081.78,6939.28,2014
1,20230614,2,10102,SELVA,AMAZONAS,CHACHAPOYAS,ASUNCION,291,151,140,0.61,33.56,14.38,47.95,2014
2,20230614,3,10103,SIERRA,AMAZONAS,CHACHAPOYAS,BALSAS,1615,299,1316,0.45,48.96,20.98,69.95,2014
3,20230614,4,10104,SIERRA,AMAZONAS,CHACHAPOYAS,CHETO,597,388,209,0.45,63.59,27.25,90.84,2014
4,20230614,5,10105,SIERRA,AMAZONAS,CHACHAPOYAS,CHILIQUIN,737,197,540,0.45,32.38,13.88,46.26,2014


In [2]:
# Verficamos los tipos de datos
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 20650 entries, 0 to 20649
Data columns (total 15 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   FECHA_CORTE       20650 non-null  int64  
 1   N_SEC             20650 non-null  int64  
 2   UBIGEO            20650 non-null  str    
 3   REG_NAT           20650 non-null  str    
 4   DEPARTAMENTO      20650 non-null  str    
 5   PROVINCIA         20650 non-null  str    
 6   DISTRITO          20650 non-null  str    
 7   POB_TOTAL         20650 non-null  int64  
 8   POB_URBANA        20650 non-null  int64  
 9   POB_RURAL         20650 non-null  int64  
 10  GPC_DOM           20650 non-null  float64
 11  QRESIDUOS_DOM     20650 non-null  float64
 12  QRESIDUOS_NO_DOM  20650 non-null  float64
 13  QRESIDUOS_MUN     20650 non-null  float64
 14  PERIODO           20650 non-null  int64  
dtypes: float64(4), int64(6), str(5)
memory usage: 2.4 MB


In [3]:
# Para trabajar con el ubigeo, necesitamos que sea un codigo de 6 dígitos
# Validamos cuantos digitos tienen los datos por cada registro
ubigeo_set = set({})
ubigeo_erroneo = list([])

for ubigeo in df["UBIGEO"]:
    if len(ubigeo)<6:
        ubigeo_erroneo.append(ubigeo)
    ubigeo_set.add(len(ubigeo))

if ubigeo_set == {6}:
    print("Todos los ubigeos en el campo UBIGEO tienen 6 dígitos")
else:
    print("Existen ubigeos con {} dígitos".format(ubigeo_set))
    print("Los ubigeos que no cumplen la cantidad de dígitos son: {}".format(ubigeo_erroneo))

# Los ubigeos con 5 dígitos son a causa que de origen se ha omitido el 0 al inicio
df["UBIGEO"] = df["UBIGEO"].str.zfill(6)

# Verificamos que la corrección fue exitosa
ubigeo_set_verificacion = set(len(ubigeo) for ubigeo in df["UBIGEO"])

if ubigeo_set_verificacion == {6}:
    print("Corrección exitosa, todos los UBIGEO tienen 6 dígitos")
    print(f"Total de ubigeos corregidos: {len(ubigeo_erroneo)}")
else:
    print("Aún existen ubigeos con dígitos incorrectos: {}".format(ubigeo_set_verificacion))

Existen ubigeos con {5, 6} dígitos
Los ubigeos que no cumplen la cantidad de dígitos son: ['10101', '10102', '10103', '10104', '10105', '10106', '10107', '10108', '10109', '10110', '10111', '10112', '10113', '10114', '10115', '10116', '10117', '10118', '10119', '10120', '10121', '10201', '10202', '10203', '10204', '10205', '10206', '10301', '10302', '10303', '10304', '10305', '10306', '10307', '10308', '10309', '10310', '10311', '10312', '10401', '10402', '10403', '10501', '10502', '10503', '10504', '10505', '10506', '10507', '10508', '10509', '10510', '10511', '10512', '10513', '10514', '10515', '10516', '10517', '10518', '10519', '10520', '10521', '10522', '10523', '10601', '10602', '10603', '10604', '10605', '10606', '10607', '10608', '10609', '10610', '10611', '10612', '10701', '10702', '10703', '10704', '10705', '10706', '10707', '20101', '20102', '20103', '20104', '20105', '20106', '20107', '20108', '20109', '20110', '20111', '20112', '20201', '20202', '20203', '20204', '20205', 

In [4]:
# El campo DISTRITO contiene en el nombre un patron numero/ al final de cada nombre por lo que debemos limpiarlo
mask_sucios = df["DISTRITO"].str.contains(r'\d*\/+', regex=True, na=False)
distritos_sucios = df[mask_sucios][["UBIGEO", "DISTRITO"]].drop_duplicates()
if distritos_sucios.empty:
    print("No se encontraron distritos con el patrón número/")
else:
    print("Se encontraron {} registro(s) con patrón número/:".format(len(distritos_sucios)))
    print(distritos_sucios.to_string(index=False))

Se encontraron 30 registro(s) con patrón número/:
UBIGEO            DISTRITO
030604        HUACCANA 17/
030612        AHUAYRO  17/
050406       SANTILLANA 9/
050413           PUTIS  9/
050501 SAN MIGUEL 11/  12/
050502            ANCO  6/
050509       SAMUGARI  10/
050512  UNIÓN PROGRESO  6/
050513   RIO MAGDALENA 10/
050514       NINABAMBA 11/
050515       PATIBAMBA 12/
080902       ECHARATE  14/
080907   KIMBIRI   15/ 16/
080910        PICHARI  18/
080915  KUMPIRUSHIATO  14/
080916    CIELO PUNCO  15/
080917         MANITEA 16/
080918 UNIÓN ASHÁNINKA 18/
090707      HUACHOCOLPA 4/
090717       SURCUBAMBA 4/
090718    TINTAY PUNCU  7/
090724          LAMBRAS 4/
090725       COCHABAMBA 7/
180107     SAN ANTONIO 19/
221001         TOCACHE  5/
221005           UCHIZA 5/
221006      SANTA LUCIA 5/
250301   PADRE ABAD 8/ 13/
250306          HUIPOCA 8/
250307       BOQUERON  13/


In [5]:
import re

# Procedemos a limpiar el campo DISTRITO
df["DISTRITO"] = (
    df["DISTRITO"]
    .str.strip()
    .str.replace(r'\s+\d+\/+.*$', '', regex=True)
    .str.replace(r'\s+', ' ', regex=True)
    .str.strip()
)

# Debido a que tenemos ubigeo podemos detectar si  hay ubigeos con más de 1 distrito asignado
def verificar_unicidad(df, col_agrupadora, col_verificar):
    verificacion = df.groupby(col_agrupadora)[col_verificar].nunique()
    casos_multiples = verificacion[verificacion > 1]

    if casos_multiples.empty:
        print("Todos los {} tienen un único {}.".format(col_agrupadora, col_verificar))
    else:
        print("Hay {} {} con más de un {}:".format(
            len(casos_multiples), col_agrupadora, col_verificar
        ))
        for grupo in casos_multiples.index:
            valores = df[df[col_agrupadora] == grupo][col_verificar].unique()
            print("  {} {}: {}".format(col_agrupadora, grupo, valores))
    
    return casos_multiples  

verificar_unicidad(df, "UBIGEO", "DISTRITO")

Hay 20 UBIGEO con más de un DISTRITO:
  UBIGEO 030407: <StringArray>
['IHUAYLLO', 'HUAYLLO']
Length: 2, dtype: str
  UBIGEO 050116: <StringArray>
['ANDRÉS AVELINO CÁCERES DORREGARAY', 'ANDRES AVELINO CACERES DORREGARAY']
Length: 2, dtype: str
  UBIGEO 050511: <StringArray>
['ORONCOY', 'ORONCCOY']
Length: 2, dtype: str
  UBIGEO 050512: <StringArray>
['UNIÓN PROGRESO', 'UNION PROGRESO']
Length: 2, dtype: str
  UBIGEO 080918: <StringArray>
['UNIÓN ASHÁNINKA', 'UNION ASHÁNINKA']
Length: 2, dtype: str
  UBIGEO 100101: <StringArray>
['HUANUCO', 'HUÁNUCO']
Length: 2, dtype: str
  UBIGEO 110301: <StringArray>
['NASCA', 'NAZCA']
Length: 2, dtype: str
  UBIGEO 120303: <StringArray>
['PICHANAQUI', 'PICHANAKI']
Length: 2, dtype: str
  UBIGEO 120501: <StringArray>
['JUNIN', 'JUNÍN']
Length: 2, dtype: str
  UBIGEO 120609: <StringArray>
['VIZCATÁN DEL ENE', 'VIZCATAN DEL ENE']
Length: 2, dtype: str
  UBIGEO 120807: <StringArray>
['SANTA BARBARA DE CARHUACAYA', 'SANTA BARBARA DE CARHUACAYAN']
Length: 

UBIGEO
030407    2
050116    2
050511    2
050512    2
080918    2
100101    2
110301    2
120303    2
120501    2
120609    2
120807    2
150121    2
150731    2
190108    2
200115    2
211210    2
230110    2
230402    2
250201    2
250401    2
Name: DISTRITO, dtype: int64

In [6]:
# Por ubigeo tenemos que regularizar los nombres, para determinar que nombre puede ser el correcto podríamos guiarnos de https://visor.geoperu.gob.pe/
# Asimismo, podríamos consultar los ubigeos de este pdf del estado peruano https://www.mef.gob.pe/contenidos/presu_publ/migl/metas/aplicativo2_meta19_24_28_30_2018.pdf
# Dicha plataforma nos indica los nombres a nivel de distritos, provincias y departamentos
# La idea es armar un diccionario para aplicar una función que regularice por los nombres
dict_distrito = {
    '100101':'HUÁNUCO',
    '110301':'NASCA',
    '120303':'PICHANAQUI',
    '120501':'JUNÍN',
    '120609':'VIZCATAN DEL ENE',
    '120807':'SANTA BARBARA DE CARHUACAYAN',
    '150121':'PUEBLO LIBRE',
    '150731':'SANTO DOMINGO DE LOS OLLEROS',
    '190108':'SAN FRANCISCO DE ASIS DE YARUSYACAN',
    '200115':'VEINTISEIS DE OCTUBRE',
    '211210':'SAN PEDRO DE PUTINA PUNCO',
    '230110':'CORONEL GREGORIO ALBARRACIN LANCHIPA',
    '230402':'HEROES ALBARRACIN',
    '250201':'RAIMONDI',
    '250301':'PADRE ABAD',
    '250401':'PURUS',
    '030407':'HUAYLLO',
    '050116':'ANDRES AVELINO CACERES DORREGARAY',
    '050501':'SAN MIGUEL',
    '050511':'ORONCCOY',
    '050512':'UNION PROGRESO',
    '080907':'KIMBIRI',
    '080918':'UNIÓN ASHÁNINKA',
}

def regularizar_distrito(ubigeo, distrito):
    # si encuentra el ubigeo devuelve el nombre correcto, si no devuelve el distrito original
    return dict_distrito.get(ubigeo, distrito)  

df["DISTRITO"] = df.apply(lambda row: regularizar_distrito(row["UBIGEO"], row["DISTRITO"]), axis=1)

# Validamos si hay más de un distrito por UBIGEO
verificar_unicidad(df, "UBIGEO", "DISTRITO")

Todos los UBIGEO tienen un único DISTRITO.


Series([], Name: DISTRITO, dtype: int64)

In [7]:
# Para trabajar con la fecha de corte necesitamos su tipo de dato y formato correcto
# Le daremos formato string a fecha de corte
df["FECHA_CORTE"] = df["FECHA_CORTE"].astype(str)

# Verificamos la longitud en el campo FECHA_CORTE, todos los registros deberían tener 8 digitos cumpliendo el formato YYYYMMDD
fechas_set = set({})

for fecha in df["FECHA_CORTE"]:
    fechas_set.add(len(fecha))

if fechas_set == {8}:
    print("Todas las fechas en FECHA_CORTE tienen 8 dígitos")
else:
    print("Existen fechas en FECHA_CORTE con longitudes distintas: {}".format(fechas_set))

# Damos el formato de fecha YYYY-MM-DD HH:MM:ss con pandas datetime; los registros que no cumplan el formato se colocarán como NaT
# En el dataset en FECHA_CORTE hay fechas con el formato YYYYMMDD y DDMMYYYY por lo que tendremos que manejar ambas situaciones
def parse_fecha(fecha):
    # Intenta formato YYYYMMDD primero
    try:
        parsed = pd.to_datetime(fecha, format="%Y%m%d")
        # Valida que el año sea razonable (evita que DDMMYYYY se parsee mal)
        if parsed.year > 1900 and parsed.year < 2100:
            return parsed
    except:
        pass
    # Si falla, intenta formato DDMMYYYY
    try:
        return pd.to_datetime(fecha, format="%d%m%Y")
    except:
        return pd.NaT

df["FECHA_CORTE"] = df["FECHA_CORTE"].apply(parse_fecha)

# Verificación de fechas que no pudieron parsearse correctamente
nats = df["FECHA_CORTE"].isna().sum()
print(f"Fechas parseadas correctamente: {len(df) - nats}")
print(f"Fechas no reconocidas (NaT): {nats}")

df["FECHA_CORTE"].head()

Todas las fechas en FECHA_CORTE tienen 8 dígitos
Fechas parseadas correctamente: 20650
Fechas no reconocidas (NaT): 0


0   2023-06-14
1   2023-06-14
2   2023-06-14
3   2023-06-14
4   2023-06-14
Name: FECHA_CORTE, dtype: datetime64[us]

In [8]:
# Con los datos geográficos, podemos potenciar el análisis obteniendo la latitud y longitud a nivel de distrito
# Existen plataformas como https://inkamaps.com/ y https://account.geodir.co/recursos/ubigeo-reniec-peru.html que nos brindan datos de geolocalización
# Consultaremos https://inkamaps.com/ para obtener las coordenadas en base al ubigeo respectivo
# https://account.geodir.co/recursos/ubigeo-reniec-peru.html

import requests
import re

# Utilizaremos la URL de inkamaps y el json de datos geográficos que brinda
url = "https://inkamaps.com/data/geodir-ubigeo-inei.json"
response = requests.get(url)
data = response.json()

# De json pasamos a un pandas dataframe
df_geografico = pd.DataFrame(data["data"])

df_geografico.head()

,ubigeo_url,distrito,provincia,departamento,poblacion,superficie
0,"<a class=""hover:text-blue-500"" href='https://m...",Chachapoyas,Chachapoyas,Amazonas,29171,153.78
1,"<a class=""hover:text-blue-500"" href='https://m...",Asuncion,Chachapoyas,Amazonas,288,25.71
2,"<a class=""hover:text-blue-500"" href='https://m...",Balsas,Chachapoyas,Amazonas,1644,357.09
3,"<a class=""hover:text-blue-500"" href='https://m...",Cheto,Chachapoyas,Amazonas,591,56.97
4,"<a class=""hover:text-blue-500"" href='https://m...",Chiliquin,Chachapoyas,Amazonas,687,143.43


In [9]:
# Las URL que redirigen a mapa tienen el siguiente formato:
# href='https://maps.geodir.co/place/ubigeo-010101-distrito-de-chachapoyas/-6.229400/-77.871400/14/b7e8
# De esa redirección podemos obtener el ubigeo, latitud y longitud

# Determinamos un patrón en la URL para obtener la latitud y longitud
patron = r'ubigeo-(\d+)[^/]*/(-?\d+\.\d+)/(-?\d+\.\d+)/'

def extraer_datos_geograficos(html):
    match = re.search(patron, html)
    # Si hay considencia con el patron definido devolvemos el ubigeo, latitud y longitud
    if match:
        return match.group(1), float(match.group(2)), float(match.group(3))
    return None, None, None

# Aplicamos la función de extraer datos geográficos
df_geografico[["ubigeo", "latitud", "longitud"]] = df_geografico["ubigeo_url"].apply(lambda x: pd.Series(extraer_datos_geograficos(x)))

# Verificamos si hay algun nulo en el dataframe de ubigeo y los tipos de datos, e imprimimos el dataframe
print(df_geografico.info())
df_geografico.head()

<class 'pandas.DataFrame'>
RangeIndex: 1874 entries, 0 to 1873
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   ubigeo_url    1874 non-null   str    
 1   distrito      1874 non-null   str    
 2   provincia     1874 non-null   str    
 3   departamento  1874 non-null   str    
 4   poblacion     1874 non-null   int64  
 5   superficie    1874 non-null   float64
 6   ubigeo        1874 non-null   str    
 7   latitud       1874 non-null   float64
 8   longitud      1874 non-null   float64
dtypes: float64(3), int64(1), str(5)
memory usage: 131.9 KB
None


,ubigeo_url,distrito,provincia,departamento,poblacion,superficie,ubigeo,latitud,longitud
0,"<a class=""hover:text-blue-500"" href='https://m...",Chachapoyas,Chachapoyas,Amazonas,29171,153.78,010101,-6.2294,-77.8714
1,"<a class=""hover:text-blue-500"" href='https://m...",Asuncion,Chachapoyas,Amazonas,288,25.71,010102,-6.0317,-77.7122
2,"<a class=""hover:text-blue-500"" href='https://m...",Balsas,Chachapoyas,Amazonas,1644,357.09,010103,-6.8375,-78.0214
3,"<a class=""hover:text-blue-500"" href='https://m...",Cheto,Chachapoyas,Amazonas,591,56.97,010104,-6.2558,-77.7003
4,"<a class=""hover:text-blue-500"" href='https://m...",Chiliquin,Chachapoyas,Amazonas,687,143.43,010105,-6.0778,-77.7392


In [10]:
# No todos los ubigeos se encontraron en esa plataforma, por lo que haremos una unión con un dataframe extra
# De esa manera tendremos un solo dataframe con puros ubigeos y latitudes y longitudes necesarias para el posterior análisis
df_geografico_ref = df_geografico[["ubigeo","latitud","longitud"]].copy()
df_geografico_extra = pd.read_excel("Diccionario Ubigeos Extra.xlsx",dtype={"ubigeo":str})
df_geografico_extra = df_geografico_extra[["ubigeo","latitud","longitud"]].copy()

df_geografico_final = (
    pd.concat([df_geografico_ref, df_geografico_extra], ignore_index=True)
    .drop_duplicates(subset="ubigeo", keep="first")
    .reset_index(drop=True)
)

print("Registros en ref:    {}".format(len(df_geografico_ref)))
print("Registros en extra:  {}".format(len(df_geografico_extra)))
print("Registros en final:  {}".format(len(df_geografico_final)))

Registros en ref:    1874
Registros en extra:  19
Registros en final:  1891


In [11]:
# Podemos guardar nuestro dataframe geográfico para futuros proyectos
df_geografico_final.to_excel("datos_geograficos_inkamaps.xlsx",index=False)

In [12]:
# Ahora nos toca añadir la latitud y longitud al dataframe principal, los campos que añadiremos serán latitud y longitud
# La llave para unir los dos dataframes sera UBIGEO para df y ubigeo para df_geografico
# Ya verificamos que ambos son str

# Primero validamos cual es la longitud del dataframe original
print("El tamaño del dataframe original es: ",df.shape)

# Hacemos la union de tipo left (izquierda) con el dataframe de ubigeo
df = df.merge(
    df_geografico_final[["ubigeo", "latitud", "longitud"]],
    left_on="UBIGEO",
    right_on="ubigeo",
    how="left"
).drop(columns=["ubigeo"])  # eliminamos la columna duplicada de la llave

# Verificamos que el merge fue exitoso
print(f"Shape del dataframe: {df.shape}")
print(f"\nNulos en latitud: {df['latitud'].isnull().sum()}")
print(f"Nulos en longitud: {df['longitud'].isnull().sum()}")
print(f"\nEjemplo del resultado en el dataframe principal:")
print(df[["UBIGEO", "latitud", "longitud"]].head())

El tamaño del dataframe original es:  (20650, 15)
Shape del dataframe: (20650, 17)

Nulos en latitud: 0
Nulos en longitud: 0

Ejemplo del resultado en el dataframe principal:
   UBIGEO  latitud  longitud
0  010101  -6.2294  -77.8714
1  010102  -6.0317  -77.7122
2  010103  -6.8375  -78.0214
3  010104  -6.2558  -77.7003
4  010105  -6.0778  -77.7392


In [13]:
# Una vez con la base trabajada podemos realizar distintos análisis con los datos que tenemos
# Procedemos a guardar nuestra base trabajada
df.to_excel("DataSet Residuos Municipales Trabajada.xlsx",index=False)